# Data Sources

In [3]:
import pyspark
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .appName("Pyspark Practice")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession lista. Version:", spark.version)

SparkSession lista. Version: 4.1.2


In [4]:
# Generamos un CSV "problematico" para mostrar las opciones
csv_data = '''id;nombre;direccion;monto;fecha
1;Soprole Vespucio;"Av. Vespucio 1737, La Florida";125000;15-03-2024
2;Tucapel Maipu;"Av. Pajaritos 5500, Maipu";87500;16-03-2024a
3;CCU Quilicur;"Av. Americo Vespucio 2050, Quilicura";200000;17-03-2024
4;Soprole Las Condes;"Av. Apoquindo 4501, Las Condes";315000;18-03-2024
5;Tucapel Concepcion;"O'Higgins 850, Concepcion";95000;19-03-2024
'''

with open("02-data/tmp/ventas_raw.csv", "w", encoding="utf-8") as f:
    f.write(csv_data)

# Verificamos
!head 02-data/tmp/ventas_raw.csv

id;nombre;direccion;monto;fecha
1;Soprole Vespucio;"Av. Vespucio 1737, La Florida";125000;15-03-2024
2;Tucapel Maipu;"Av. Pajaritos 5500, Maipu";87500;16-03-2024a
3;CCU Quilicur;"Av. Americo Vespucio 2050, Quilicura";200000;17-03-2024
4;Soprole Las Condes;"Av. Apoquindo 4501, Las Condes";315000;18-03-2024
5;Tucapel Concepcion;"O'Higgins 850, Concepcion";95000;19-03-2024


In [5]:
# Intento 1: con los defaults (delimitador coma)
df_malo = spark.read.csv("02-data/tmp/ventas_raw.csv", header=True)
df_malo.show(truncate=False)
df_malo.printSchema()

+-----------------------------------------+
|id;nombre;direccion;monto;fecha          |
+-----------------------------------------+
|1;Soprole Vespucio;"Av. Vespucio 1737    |
|2;Tucapel Maipu;"Av. Pajaritos 5500      |
|3;CCU Quilicur;"Av. Americo Vespucio 2050|
|4;Soprole Las Condes;"Av. Apoquindo 4501 |
|5;Tucapel Concepcion;"O'Higgins 850      |
+-----------------------------------------+

root
 |-- id;nombre;direccion;monto;fecha: string (nullable = true)



In [6]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType
)

# Definimos el schema una vez, lo reutilizamos siempre
schema_ventas = StructType([
    StructField("id",        IntegerType(), nullable=False),
    StructField("nombre",    StringType(),  nullable=True),
    StructField("direccion", StringType(),  nullable=True),
    StructField("monto",     DoubleType(),  nullable=True),
    StructField("fecha",     DateType(),    nullable=True),
])

df_csv = (
    spark.read
    .schema(schema_ventas)                # schema explicito = mas rapido y seguro
    .option("delimiter", ";")             # nuestro separador
    .option("header", "true")             # primera fila es header
    .option("quote", '"')                 # las comillas envuelven valores con comas
    .option("dateFormat", "dd-MM-yyyy")   # fecha en formato chileno
    .csv("02-data/tmp/ventas_raw.csv")
)

df_csv.show(truncate=False)
df_csv.printSchema()

+---+------------------+------------------------------------+--------+----------+
|id |nombre            |direccion                           |monto   |fecha     |
+---+------------------+------------------------------------+--------+----------+
|1  |Soprole Vespucio  |Av. Vespucio 1737, La Florida       |125000.0|2024-03-15|
|2  |Tucapel Maipu     |Av. Pajaritos 5500, Maipu           |87500.0 |NULL      |
|3  |CCU Quilicur      |Av. Americo Vespucio 2050, Quilicura|200000.0|2024-03-17|
|4  |Soprole Las Condes|Av. Apoquindo 4501, Las Condes      |315000.0|2024-03-18|
|5  |Tucapel Concepcion|O'Higgins 850, Concepcion           |95000.0 |2024-03-19|
+---+------------------+------------------------------------+--------+----------+

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- direccion: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- fecha: date (nullable = true)



In [7]:
import json

sucursales = [
    {
        "id": 1,
        "nombre": "Sucursal Vespucio",
        "direccion": {"comuna": "La Florida", "region": "Metropolitana", "calle": "Av. Vespucio 1737"},
        "servicios": ["bodega", "delivery", "estacionamiento"]
    },
    {
        "id": 2,
        "nombre": "Sucursal Concepcion Centro",
        "direccion": {"comuna": "Concepcion", "region": "Biobio", "calle": "O'Higgins 850"},
        "servicios": ["bodega", "delivery"]
    },
    {
        "id": 3,
        "nombre": "Sucursal Mall Plaza Antofagasta",
        "direccion": {"comuna": "Antofagasta", "region": "Antofagasta", "calle": "Av. Balmaceda 2355"},
        "servicios": ["bodega", "delivery", "estacionamiento", "cafeteria"]
    },
]

# JSON Lines (un objeto por linea) - es lo que prefiere Spark por defecto
with open("02-data/tmp/sucursales.jsonl", "w", encoding="utf-8") as f:
    for s in sucursales:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

# Tambien guardamos un JSON "normal" (array completo en una linea) para mostrar multiLine
with open("02-data/tmp/sucursales_array.json", "w", encoding="utf-8") as f:
    json.dump(sucursales, f, ensure_ascii=False, indent=2)

!cat 02-data/tmp/sucursales.jsonl
print("---")
!cat 02-data/tmp/sucursales_array.json

{"id": 1, "nombre": "Sucursal Vespucio", "direccion": {"comuna": "La Florida", "region": "Metropolitana", "calle": "Av. Vespucio 1737"}, "servicios": ["bodega", "delivery", "estacionamiento"]}
{"id": 2, "nombre": "Sucursal Concepcion Centro", "direccion": {"comuna": "Concepcion", "region": "Biobio", "calle": "O'Higgins 850"}, "servicios": ["bodega", "delivery"]}
{"id": 3, "nombre": "Sucursal Mall Plaza Antofagasta", "direccion": {"comuna": "Antofagasta", "region": "Antofagasta", "calle": "Av. Balmaceda 2355"}, "servicios": ["bodega", "delivery", "estacionamiento", "cafeteria"]}
---
[
  {
    "id": 1,
    "nombre": "Sucursal Vespucio",
    "direccion": {
      "comuna": "La Florida",
      "region": "Metropolitana",
      "calle": "Av. Vespucio 1737"
    },
    "servicios": [
      "bodega",
      "delivery",
      "estacionamiento"
    ]
  },
  {
    "id": 2,
    "nombre": "Sucursal Concepcion Centro",
    "direccion": {
      "comuna": "Concepcion",
      "region": "Biobio",
      "ca

In [8]:
# Caso 1: JSON Lines (default)
df_suc = spark.read.json("02-data/tmp/sucursales.jsonl")
df_suc.printSchema()
df_suc.show(truncate=False)

root
 |-- direccion: struct (nullable = true)
 |    |-- calle: string (nullable = true)
 |    |-- comuna: string (nullable = true)
 |    |-- region: string (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- servicios: array (nullable = true)
 |    |-- element: string (containsNull = true)

+----------------------------------------------+---+-------------------------------+----------------------------------------------+
|direccion                                     |id |nombre                         |servicios                                     |
+----------------------------------------------+---+-------------------------------+----------------------------------------------+
|{Av. Vespucio 1737, La Florida, Metropolitana}|1  |Sucursal Vespucio              |[bodega, delivery, estacionamiento]           |
|{O'Higgins 850, Concepcion, Biobio}           |2  |Sucursal Concepcion Centro     |[bodega, delivery]                            |
|{Av. B

In [9]:
# Caso 2: JSON con array completo (multiLine)
df_suc_arr = spark.read.option("multiLine", "true").json("02-data/tmp/sucursales_array.json")
df_suc_arr.show(truncate=False)

+----------------------------------------------+---+-------------------------------+----------------------------------------------+
|direccion                                     |id |nombre                         |servicios                                     |
+----------------------------------------------+---+-------------------------------+----------------------------------------------+
|{Av. Vespucio 1737, La Florida, Metropolitana}|1  |Sucursal Vespucio              |[bodega, delivery, estacionamiento]           |
|{O'Higgins 850, Concepcion, Biobio}           |2  |Sucursal Concepcion Centro     |[bodega, delivery]                            |
|{Av. Balmaceda 2355, Antofagasta, Antofagasta}|3  |Sucursal Mall Plaza Antofagasta|[bodega, delivery, estacionamiento, cafeteria]|
+----------------------------------------------+---+-------------------------------+----------------------------------------------+



In [10]:
# Acceder a campos anidados
from pyspark.sql.functions import col

(df_suc
    .select(
        col("nombre"),
        col("direccion.comuna").alias("comuna"),
        col("direccion.region").alias("region"),
        col("servicios")
    )
    .show(truncate=False)
)

+-------------------------------+-----------+-------------+----------------------------------------------+
|nombre                         |comuna     |region       |servicios                                     |
+-------------------------------+-----------+-------------+----------------------------------------------+
|Sucursal Vespucio              |La Florida |Metropolitana|[bodega, delivery, estacionamiento]           |
|Sucursal Concepcion Centro     |Concepcion |Biobio       |[bodega, delivery]                            |
|Sucursal Mall Plaza Antofagasta|Antofagasta|Antofagasta  |[bodega, delivery, estacionamiento, cafeteria]|
+-------------------------------+-----------+-------------+----------------------------------------------+



In [11]:
# Escribir Parquet
df_csv.write.mode("overwrite").parquet("02-data/tmp/ventas_parquet")

# Spark escribe una carpeta, no un solo archivo (un archivo por particion)
!ls -lh 02-data/tmp/ventas_parquet/

total 4,0K
-rw-r--r-- 1 carlos carlos 1,9K may 27 10:35 part-00000-93f805a0-b7ab-4a16-9bfd-5a0135795868-c000.snappy.parquet
-rw-r--r-- 1 carlos carlos    0 may 27 10:35 _SUCCESS


In [12]:
# Comparar tamanos: CSV vs Parquet
!echo "--- CSV original ---"
!ls -lh 02-data/tmp/ventas_raw.csv
!echo "--- Parquet ---"
!du -sh 02-data/tmp/ventas_parquet/

--- CSV original ---
-rw-rw-r-- 1 carlos carlos 373 may 27 10:35 02-data/tmp/ventas_raw.csv
--- Parquet ---
16K	02-data/tmp/ventas_parquet/


In [13]:
df_parquet = spark.read.parquet("02-data/tmp/ventas_parquet")
df_parquet.printSchema()
df_parquet.show()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- direccion: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- fecha: date (nullable = true)

+---+------------------+--------------------+--------+----------+
| id|            nombre|           direccion|   monto|     fecha|
+---+------------------+--------------------+--------+----------+
|  1|  Soprole Vespucio|Av. Vespucio 1737...|125000.0|2024-03-15|
|  2|     Tucapel Maipu|Av. Pajaritos 550...| 87500.0|      NULL|
|  3|      CCU Quilicur|Av. Americo Vespu...|200000.0|2024-03-17|
|  4|Soprole Las Condes|Av. Apoquindo 450...|315000.0|2024-03-18|
|  5|Tucapel Concepcion|O'Higgins 850, Co...| 95000.0|2024-03-19|
+---+------------------+--------------------+--------+----------+



## Dataset real

In [14]:
# Descarga del Parquet (~50 MB, toma menos de 1 minuto en Colab)
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O 02-data/tmp/taxi_2024_01.parquet
!ls -lh 02-data/tmp/taxi_2024_01.parquet

-rw-rw-r-- 1 carlos carlos 48M mar 21  2024 02-data/tmp/taxi_2024_01.parquet


In [15]:
df_taxi = spark.read.parquet("02-data/tmp/taxi_2024_01.parquet")

# Schema (rapido, no escanea los datos)
df_taxi.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [16]:
print("Filas totales:", df_taxi.count())

Filas totales: 2964624


In [17]:
df_taxi.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [18]:
# OJO: esto puede tomar 1-2 minutos porque CSV es mucho mas pesado
import time

t0 = time.time()
df_taxi.coalesce(1).write.mode("overwrite").option("header", "true").csv("02-data/tmp/taxi_csv")
print(f"Escritura CSV: {time.time()-t0:.1f}s")

!du -sh 02-data/tmp/taxi_csv/
!du -sh 02-data/tmp/taxi_2024_01.parquet

Escritura CSV: 9.9s
311M	02-data/tmp/taxi_csv/
48M	02-data/tmp/taxi_2024_01.parquet


In [19]:
# Tiempo lectura Parquet
t0 = time.time()
n1 = spark.read.parquet("02-data/tmp/taxi_2024_01.parquet").count()
t_parquet = time.time() - t0

# Tiempo lectura CSV (sin schema explicito = lo peor)
t0 = time.time()
n2 = spark.read.option("header", "true").csv("02-data/tmp/taxi_csv").count()
t_csv = time.time() - t0

print(f"Parquet: {n1:,} filas en {t_parquet:.2f}s")
print(f"CSV:     {n2:,} filas en {t_csv:.2f}s")
print(f"Parquet es {t_csv/t_parquet:.1f}x mas rapido")

Parquet: 2,964,624 filas en 0.30s
CSV:     2,964,624 filas en 1.43s
Parquet es 4.8x mas rapido


# Transformaciones avanzadas

In [20]:
from pyspark.sql.functions import (
    col, year, month, dayofweek, hour, date_format,
    unix_timestamp, round as f_round
)

# Extraer componentes de la fecha de pickup
df_fechas = df_taxi.select(
    col("tpep_pickup_datetime"),
    year("tpep_pickup_datetime").alias("anio"),
    month("tpep_pickup_datetime").alias("mes"),
    dayofweek("tpep_pickup_datetime").alias("dia_semana"),   # 1=Domingo, 7=Sabado
    hour("tpep_pickup_datetime").alias("hora"),
    date_format("tpep_pickup_datetime", "EEEE").alias("dia_nombre")  # "Monday", etc.
)

df_fechas.show(5, truncate=False)

+--------------------+----+---+----------+----+----------+
|tpep_pickup_datetime|anio|mes|dia_semana|hora|dia_nombre|
+--------------------+----+---+----------+----+----------+
|2024-01-01 00:57:55 |2024|1  |2         |0   |Monday    |
|2024-01-01 00:03:00 |2024|1  |2         |0   |Monday    |
|2024-01-01 00:17:06 |2024|1  |2         |0   |Monday    |
|2024-01-01 00:36:38 |2024|1  |2         |0   |Monday    |
|2024-01-01 00:46:51 |2024|1  |2         |0   |Monday    |
+--------------------+----+---+----------+----+----------+
only showing top 5 rows


In [21]:
df_duracion = df_taxi.withColumn(
    "duracion_min",
    f_round(
        (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
        2
    )
)

df_duracion.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duracion_min").show(5, truncate=False)

+--------------------+---------------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|duracion_min|
+--------------------+---------------------+------------+
|2024-01-01 00:57:55 |2024-01-01 01:17:43  |19.8        |
|2024-01-01 00:03:00 |2024-01-01 00:09:36  |6.6         |
|2024-01-01 00:17:06 |2024-01-01 00:35:01  |17.92       |
|2024-01-01 00:36:38 |2024-01-01 00:44:56  |8.3         |
|2024-01-01 00:46:51 |2024-01-01 00:52:57  |6.1         |
+--------------------+---------------------+------------+
only showing top 5 rows


In [22]:
df_limpio = df_duracion.filter(
    (col("duracion_min") > 0) & (col("duracion_min") < 240) &
    (col("trip_distance") > 0) & (col("trip_distance") < 100) &
    (col("total_amount") > 0)
)

print("Antes:", df_taxi.count(), "filas")
print("Despues:", df_limpio.count(), "filas")
print("Removidos:", df_taxi.count() - df_limpio.count())

Antes: 2964624 filas


Despues: 2870011 filas


Removidos: 94613


In [23]:
from pyspark.sql.functions import (
    concat, concat_ws, substring, length, lower, upper, initcap, trim,
    regexp_extract, regexp_replace, split, lit
)

df_suc.select(
    col("nombre"),
    upper(col("nombre")).alias("upper"),
    lower(col("nombre")).alias("lower"),
    length(col("nombre")).alias("largo"),
    substring(col("nombre"), 1, 8).alias("primeros_8")
).show(truncate=False)

+-------------------------------+-------------------------------+-------------------------------+-----+----------+
|nombre                         |upper                          |lower                          |largo|primeros_8|
+-------------------------------+-------------------------------+-------------------------------+-----+----------+
|Sucursal Vespucio              |SUCURSAL VESPUCIO              |sucursal vespucio              |17   |Sucursal  |
|Sucursal Concepcion Centro     |SUCURSAL CONCEPCION CENTRO     |sucursal concepcion centro     |26   |Sucursal  |
|Sucursal Mall Plaza Antofagasta|SUCURSAL MALL PLAZA ANTOFAGASTA|sucursal mall plaza antofagasta|31   |Sucursal  |
+-------------------------------+-------------------------------+-------------------------------+-----+----------+



In [24]:
# Combinar comuna y region en una sola columna "ubicacion"
df_suc.select(
    col("nombre"),
    concat_ws(", ", col("direccion.comuna"), col("direccion.region")).alias("ubicacion")
).show(truncate=False)

+-------------------------------+-------------------------+
|nombre                         |ubicacion                |
+-------------------------------+-------------------------+
|Sucursal Vespucio              |La Florida, Metropolitana|
|Sucursal Concepcion Centro     |Concepcion, Biobio       |
|Sucursal Mall Plaza Antofagasta|Antofagasta, Antofagasta |
+-------------------------------+-------------------------+



In [25]:
# Extraer el numero de la calle de la direccion
# Patron: cualquier cosa, luego un numero, opcional ", "
df_calles = df_suc.select(
    col("direccion.calle").alias("calle"),
    regexp_extract(col("direccion.calle"), r"(\d+)", 1).alias("numero")
)
df_calles.show(truncate=False)

+------------------+------+
|calle             |numero|
+------------------+------+
|Av. Vespucio 1737 |1737  |
|O'Higgins 850     |850   |
|Av. Balmaceda 2355|2355  |
+------------------+------+



In [26]:
df_split = df_suc.select(
    col("direccion.calle").alias("calle"),
    split(col("direccion.calle"), " ").alias("partes"),
    split(col("direccion.calle"), " ").getItem(0).alias("primera_palabra")
)
df_split.show(truncate=False)

+------------------+----------------------+---------------+
|calle             |partes                |primera_palabra|
+------------------+----------------------+---------------+
|Av. Vespucio 1737 |[Av., Vespucio, 1737] |Av.            |
|O'Higgins 850     |[O'Higgins, 850]      |O'Higgins      |
|Av. Balmaceda 2355|[Av., Balmaceda, 2355]|Av.            |
+------------------+----------------------+---------------+



In [27]:
from pyspark.sql.functions import size, array_contains, explode, sort_array

df_suc.select(
    col("nombre"),
    col("servicios"),
    size(col("servicios")).alias("cant_servicios"),
    array_contains(col("servicios"), "cafeteria").alias("tiene_cafeteria")
).show(truncate=False)

+-------------------------------+----------------------------------------------+--------------+---------------+
|nombre                         |servicios                                     |cant_servicios|tiene_cafeteria|
+-------------------------------+----------------------------------------------+--------------+---------------+
|Sucursal Vespucio              |[bodega, delivery, estacionamiento]           |3             |false          |
|Sucursal Concepcion Centro     |[bodega, delivery]                            |2             |false          |
|Sucursal Mall Plaza Antofagasta|[bodega, delivery, estacionamiento, cafeteria]|4             |true           |
+-------------------------------+----------------------------------------------+--------------+---------------+



In [28]:
# Una fila por (sucursal, servicio)
df_exploded = df_suc.select(
    col("nombre"),
    explode(col("servicios")).alias("servicio")
)

df_exploded.show(truncate=False)
print(f"Original: {df_suc.count()} filas, Exploded: {df_exploded.count()} filas")

+-------------------------------+---------------+
|nombre                         |servicio       |
+-------------------------------+---------------+
|Sucursal Vespucio              |bodega         |
|Sucursal Vespucio              |delivery       |
|Sucursal Vespucio              |estacionamiento|
|Sucursal Concepcion Centro     |bodega         |
|Sucursal Concepcion Centro     |delivery       |
|Sucursal Mall Plaza Antofagasta|bodega         |
|Sucursal Mall Plaza Antofagasta|delivery       |
|Sucursal Mall Plaza Antofagasta|estacionamiento|
|Sucursal Mall Plaza Antofagasta|cafeteria      |
+-------------------------------+---------------+

Original: 3 filas, Exploded: 9 filas


In [29]:
df_exploded.groupBy("servicio").count().orderBy(col("count").desc()).show()

+---------------+-----+
|       servicio|count|
+---------------+-----+
|         bodega|    3|
|       delivery|    3|
|estacionamiento|    2|
|      cafeteria|    1|
+---------------+-----+



In [30]:
from pyspark.sql.functions import count, hour, dayofweek

df_matriz = (
    df_limpio
    .withColumn("hora", hour("tpep_pickup_datetime"))
    .withColumn("dia_semana", dayofweek("tpep_pickup_datetime"))
    .groupBy("hora")
    .pivot("dia_semana", [1, 2, 3, 4, 5, 6, 7])
    .agg(count("hora"))
    .orderBy("hora")
)

df_matriz.show(24, truncate=False)

+----+-----+-----+-----+-----+-----+-----+-----+
|hora|1    |2    |3    |4    |5    |6    |7    |
+----+-----+-----+-----+-----+-----+-----+-----+
|0   |19837|11323|6071 |6369 |5400 |8432 |17829|
|1   |15647|9029 |2466 |2542 |2500 |4598 |13757|
|2   |11524|6879 |1217 |1564 |1399 |2517 |9888 |
|3   |7216 |5339 |878  |1051 |1004 |1648 |5811 |
|4   |3687 |3904 |1101 |1198 |1018 |1365 |3018 |
|5   |1554 |3512 |2926 |3082 |2430 |2459 |1532 |
|6   |2476 |6738 |7509 |7682 |6283 |5985 |2766 |
|7   |4019 |12192|16508|17444|13646|12566|4511 |
|8   |6369 |16417|22202|24149|19694|16862|7851 |
|9   |10132|17679|23037|24639|20079|17791|12244|
|10  |14888|18589|23395|23671|19715|19096|16001|
|11  |18291|20054|24851|24931|20157|19320|19092|
|12  |20674|22228|25194|26866|21470|20421|23047|
|13  |21756|23082|26059|27525|21996|21587|23263|
|14  |21751|25752|28844|29746|24242|24054|23551|
|15  |20960|26362|29722|30744|26168|24844|25106|
|16  |21248|25715|29060|30569|26380|24958|26944|
|17  |20722|27311|33

In [31]:
from pyspark.sql.functions import avg, count

(df_limpio
    .groupBy(hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        f_round(avg("total_amount"), 2).alias("promedio_usd"),
        count("*").alias("cantidad")
    )
    .orderBy(col("promedio_usd").desc())
    .show(24))

+----+------------+--------+
|hora|promedio_usd|cantidad|
+----+------------+--------+
|   5|       37.55|   17495|
|   4|       32.37|   15291|
|   6|       30.11|   39439|
|  16|       29.74|  184874|
|  23|       29.55|  104703|
|   0|       28.54|   75261|
|  22|       28.35|  138350|
|  17|       28.13|  200283|
|  14|       27.65|  177940|
|  19|       27.64|  178830|
|  21|       27.44|  155965|
|  15|       27.39|  183906|
|  20|       27.27|  155569|
|  18|       26.81|  206381|
|   3|       26.68|   22947|
|   7|        26.5|   80886|
|  13|       26.49|  165268|
|  10|       26.02|  135355|
|   9|       25.89|  125601|
|   1|       25.85|   50539|
|  12|       25.71|  159900|
|   8|       25.52|  113544|
|  11|       25.46|  146696|
|   2|       24.55|   34988|
+----+------------+--------+



In [32]:
# Espacio para que los alumnos prueben
from pyspark.sql.functions import avg, count

(df_limpio
    .groupBy(hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        f_round(avg("total_amount"), 2).alias("promedio_usd"),
        count("*").alias("cantidad")
    )
    .orderBy(col("promedio_usd").desc())
    .show(24))

+----+------------+--------+
|hora|promedio_usd|cantidad|
+----+------------+--------+
|   5|       37.55|   17495|
|   4|       32.37|   15291|
|   6|       30.11|   39439|
|  16|       29.74|  184874|
|  23|       29.55|  104703|
|   0|       28.54|   75261|
|  22|       28.35|  138350|
|  17|       28.13|  200283|
|  14|       27.65|  177940|
|  19|       27.64|  178830|
|  21|       27.44|  155965|
|  15|       27.39|  183906|
|  20|       27.27|  155569|
|  18|       26.81|  206381|
|   3|       26.68|   22947|
|   7|        26.5|   80886|
|  13|       26.49|  165268|
|  10|       26.02|  135355|
|   9|       25.89|  125601|
|   1|       25.85|   50539|
|  12|       25.71|  159900|
|   8|       25.52|  113544|
|  11|       25.46|  146696|
|   2|       24.55|   34988|
+----+------------+--------+



# Spark SQL, UDF, Windows avanzado y escritura particionada

In [33]:
# Registrar el dataset como vista SQL, “Toma este DataFrame y regístralo como una tabla temporal llamada taxi”
df_limpio.createOrReplaceTempView("taxi")

# Ya podemos hacer queries SQL
spark.sql("SELECT COUNT(*) AS total FROM taxi").show()

+-------+
|  total|
+-------+
|2870011|
+-------+



In [34]:
spark.sql("""
    SELECT
        HOUR(tpep_pickup_datetime) AS hora,
        COUNT(*) AS viajes,
        ROUND(AVG(total_amount), 2) AS promedio_usd,
        ROUND(AVG(trip_distance), 2) AS distancia_mi
    FROM taxi
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY hora
""").show(24)

+----+------+------------+------------+
|hora|viajes|promedio_usd|distancia_mi|
+----+------+------------+------------+
|   0| 75261|       28.54|        3.85|
|   1| 50539|       25.85|        3.26|
|   2| 34988|       24.55|        3.04|
|   3| 22947|       26.68|        3.51|
|   4| 15291|       32.37|        4.86|
|   5| 17495|       37.55|        6.23|
|   6| 39439|       30.11|        4.72|
|   7| 80886|        26.5|        3.56|
|   8|113544|       25.52|        3.06|
|   9|125601|       25.89|        3.05|
|  10|135355|       26.02|        3.05|
|  11|146696|       25.46|        2.91|
|  12|159900|       25.71|        2.98|
|  13|165268|       26.49|        3.16|
|  14|177940|       27.65|        3.34|
|  15|183906|       27.39|        3.27|
|  16|184874|       29.74|         3.4|
|  17|200283|       28.13|        3.05|
|  18|206381|       26.81|        2.86|
|  19|178830|       27.64|        3.16|
|  20|155569|       27.27|        3.36|
|  21|155965|       27.44|        3.46|


In [35]:
(df_limpio
    .groupBy(hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        count("*").alias("viajes"),
        f_round(avg("total_amount"), 2).alias("promedio_usd"),
        f_round(avg("trip_distance"), 2).alias("distancia_mi")
    )
    .orderBy("hora")
    .show(24))

+----+------+------------+------------+
|hora|viajes|promedio_usd|distancia_mi|
+----+------+------------+------------+
|   0| 75261|       28.54|        3.85|
|   1| 50539|       25.85|        3.26|
|   2| 34988|       24.55|        3.04|
|   3| 22947|       26.68|        3.51|
|   4| 15291|       32.37|        4.86|
|   5| 17495|       37.55|        6.23|
|   6| 39439|       30.11|        4.72|
|   7| 80886|        26.5|        3.56|
|   8|113544|       25.52|        3.06|
|   9|125601|       25.89|        3.05|
|  10|135355|       26.02|        3.05|
|  11|146696|       25.46|        2.91|
|  12|159900|       25.71|        2.98|
|  13|165268|       26.49|        3.16|
|  14|177940|       27.65|        3.34|
|  15|183906|       27.39|        3.27|
|  16|184874|       29.74|         3.4|
|  17|200283|       28.13|        3.05|
|  18|206381|       26.81|        2.86|
|  19|178830|       27.64|        3.16|
|  20|155569|       27.27|        3.36|
|  21|155965|       27.44|        3.46|


In [36]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def categorizar_viaje(distancia, duracion, total):
    if distancia is None or duracion is None or total is None:
        return "desconocido"
    if distancia < 2 and duracion < 10:
        return "corto"
    if distancia < 10 and total < 30:
        return "medio"
    if distancia >= 10 or total >= 50:
        return "largo"
    return "medio"

# Registrar como UDF tipada
categorizar_udf = udf(categorizar_viaje, StringType())

df_categorizado = df_limpio.withColumn(
    "categoria",
    categorizar_udf(col("trip_distance"), col("duracion_min"), col("total_amount"))
)

df_categorizado.groupBy("categoria").count().orderBy(col("count").desc()).show()

+---------+-------+
|categoria|  count|
+---------+-------+
|    medio|1438085|
|    corto|1118797|
|    largo| 313129|
+---------+-------+



In [37]:
from pyspark.sql.functions import when

df_cat_builtin = df_limpio.withColumn(
    "categoria",
    when((col("trip_distance") < 2) & (col("duracion_min") < 10), "corto")
    .when((col("trip_distance") < 10) & (col("total_amount") < 30), "medio")
    .when((col("trip_distance") >= 10) | (col("total_amount") >= 50), "largo")
    .otherwise("medio")
)

df_cat_builtin.groupBy("categoria").count().orderBy(col("count").desc()).show()

+---------+-------+
|categoria|  count|
+---------+-------+
|    medio|1438085|
|    corto|1118797|
|    largo| 313129|
+---------+-------+



In [38]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as f_sum

# Primero, agregamos los ingresos por hora
df_por_hora = (
    df_limpio
    .groupBy(hour("tpep_pickup_datetime").alias("hora"))
    .agg(f_sum("total_amount").alias("ingreso_hora"))
    .orderBy("hora")
)

df_por_hora.show(24)

+----+------------------+
|hora|      ingreso_hora|
+----+------------------+
|   0|2148004.7799999854|
|   1| 1306251.840000008|
|   2| 859006.7500000003|
|   3|  612173.459999999|
|   4|495000.02999999956|
|   5| 656921.2400000013|
|   6|  1187607.18000001|
|   7|  2143447.55999997|
|   8| 2897398.169999935|
|   9|3251461.7899999525|
|  10| 3521871.539999957|
|  11| 3735021.789999959|
|  12|4110938.3699999265|
|  13| 4378374.989999916|
|  14| 4919352.669999873|
|  15| 5037444.129999887|
|  16| 5497985.209999836|
|  17|  5633969.60999996|
|  18| 5533835.380000043|
|  19| 4942106.309999996|
|  20| 4243121.389999876|
|  21| 4279252.389999883|
|  22|3922294.6999999415|
|  23| 3093777.219999957|
+----+------------------+



In [39]:
# Window que acumula desde el inicio hasta la fila actual
w_acum = Window.orderBy("hora").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_acumulado = df_por_hora.withColumn(
    "ingreso_acumulado",
    f_round(f_sum("ingreso_hora").over(w_acum), 2)
)

df_acumulado.show(24)

+----+------------------+-----------------+
|hora|      ingreso_hora|ingreso_acumulado|
+----+------------------+-----------------+
|   0|2148004.7799999854|       2148004.78|
|   1| 1306251.840000008|       3454256.62|
|   2| 859006.7500000003|       4313263.37|
|   3|  612173.459999999|       4925436.83|
|   4|495000.02999999956|       5420436.86|
|   5| 656921.2400000013|        6077358.1|
|   6|  1187607.18000001|       7264965.28|
|   7|  2143447.55999997|       9408412.84|
|   8| 2897398.169999935|    1.230581101E7|
|   9|3251461.7899999525|     1.55572728E7|
|  10| 3521871.539999957|    1.907914434E7|
|  11| 3735021.789999959|    2.281416613E7|
|  12|4110938.3699999265|     2.69251045E7|
|  13| 4378374.989999916|    3.130347949E7|
|  14| 4919352.669999873|    3.622283216E7|
|  15| 5037444.129999887|    4.126027629E7|
|  16| 5497985.209999836|     4.67582615E7|
|  17|  5633969.60999996|    5.239223111E7|
|  18| 5533835.380000043|    5.792606649E7|
|  19| 4942106.309999996|     6.

In [40]:
from pyspark.sql.functions import avg as f_avg

w_movil = Window.orderBy("hora").rowsBetween(-2, 0)  # 2 anteriores + actual

df_movil = df_por_hora.withColumn(
    "promedio_movil_3h",
    f_round(f_avg("ingreso_hora").over(w_movil), 2)
)

df_movil.show(24)

+----+------------------+-----------------+
|hora|      ingreso_hora|promedio_movil_3h|
+----+------------------+-----------------+
|   0|2148004.7799999854|       2148004.78|
|   1| 1306251.840000008|       1727128.31|
|   2| 859006.7500000003|       1437754.46|
|   3|  612173.459999999|        925810.68|
|   4|495000.02999999956|        655393.41|
|   5| 656921.2400000013|        588031.58|
|   6|  1187607.18000001|        779842.82|
|   7|  2143447.55999997|       1329325.33|
|   8| 2897398.169999935|       2076150.97|
|   9|3251461.7899999525|       2764102.51|
|  10| 3521871.539999957|       3223577.17|
|  11| 3735021.789999959|       3502785.04|
|  12|4110938.3699999265|       3789277.23|
|  13| 4378374.989999916|       4074778.38|
|  14| 4919352.669999873|       4469555.34|
|  15| 5037444.129999887|        4778390.6|
|  16| 5497985.209999836|        5151594.0|
|  17|  5633969.60999996|       5389799.65|
|  18| 5533835.380000043|        5555263.4|
|  19| 4942106.309999996|       

In [41]:
from pyspark.sql.functions import dayofmonth

df_para_particionar = df_limpio.withColumn(
    "dia",
    dayofmonth("tpep_pickup_datetime")
)

# Escribimos particionado por dia
(df_para_particionar
    .write
    .mode("overwrite")
    .partitionBy("dia")
    .parquet("02-data/tmp/taxi_particionado"))

# Ver estructura de carpetas
!ls 02-data/tmp/taxi_particionado/ | head -10

dia=1
dia=10
dia=11
dia=12
dia=13
dia=14
dia=15
dia=16
dia=17
dia=18


In [42]:
# Leer todo
df_todo = spark.read.parquet("02-data/tmp/taxi_particionado")
print(f"Total: {df_todo.count():,} filas")

# Leer solo el dia 15 - Spark solo abre dia=15/
df_dia_15 = spark.read.parquet("02-data/tmp/taxi_particionado").filter(col("dia") == 15)
print(f"Dia 15: {df_dia_15.count():,} filas")

Total: 2,870,011 filas
Dia 15: 74,797 filas


## Informe ejecutivo del mes

In [43]:
# 1. Total mes
from pyspark.sql.functions import countDistinct, sum as f_sum_

resumen = df_limpio.agg(
    count("*").alias("total_viajes"),
    f_round(f_sum_("total_amount"), 0).alias("ingreso_total_usd"),
    f_round(avg("total_amount"), 2).alias("ticket_promedio"),
    f_round(avg("trip_distance"), 2).alias("distancia_promedio_mi"),
    f_round(avg("duracion_min"), 1).alias("duracion_promedio_min")
)

resumen.show(truncate=False)

+------------+-----------------+---------------+---------------------+---------------------+
|total_viajes|ingreso_total_usd|ticket_promedio|distancia_promedio_mi|duracion_promedio_min|
+------------+-----------------+---------------+---------------------+---------------------+
|2870011     |7.8406619E7      |27.32          |3.29                 |15.0                 |
+------------+-----------------+---------------+---------------------+---------------------+



In [44]:
# 2. Top 5 horas con mas ingreso
(df_limpio
    .groupBy(hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        count("*").alias("viajes"),
        f_round(f_sum_("total_amount"), 0).alias("ingreso_usd")
    )
    .orderBy(col("ingreso_usd").desc())
    .show(5))

+----+------+-----------+
|hora|viajes|ingreso_usd|
+----+------+-----------+
|  17|200283|  5633970.0|
|  18|206381|  5533835.0|
|  16|184874|  5497985.0|
|  15|183906|  5037444.0|
|  19|178830|  4942106.0|
+----+------+-----------+
only showing top 5 rows


In [45]:
# 3. Distribucion de pago
# payment_type: 1=Credit, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided
df_pago = (df_limpio
    .groupBy("payment_type")
    .agg(count("*").alias("cantidad"),
         f_round(f_sum_("total_amount"), 0).alias("monto_total"))
    .orderBy(col("cantidad").desc())
)

df_pago.createOrReplaceTempView("pago")

spark.sql("""
    SELECT
        CASE payment_type
            WHEN 1 THEN 'Tarjeta'
            WHEN 2 THEN 'Efectivo'
            WHEN 3 THEN 'Sin cargo'
            WHEN 4 THEN 'Disputa'
            ELSE 'Otro'
        END AS metodo,
        cantidad,
        monto_total,
        ROUND(100.0 * cantidad / SUM(cantidad) OVER (), 1) AS porc_viajes
    FROM pago
    ORDER BY cantidad DESC
""").show(truncate=False)

+---------+--------+-----------+-----------+
|metodo   |cantidad|monto_total|porc_viajes|
+---------+--------+-----------+-----------+
|Tarjeta  |2297039 |6.4481957E7|80.0       |
|Efectivo |422311  |1.0058919E7|14.7       |
|Otro     |117253  |3059307.0  |4.1        |
|Disputa  |22800   |571004.0   |0.8        |
|Sin cargo|10608   |235431.0   |0.4        |
+---------+--------+-----------+-----------+



In [46]:
# 4. Top 10 zonas de origen
(df_limpio
    .groupBy("PULocationID")
    .agg(count("*").alias("viajes_salida"))
    .orderBy(col("viajes_salida").desc())
    .show(10))

+------------+-------------+
|PULocationID|viajes_salida|
+------------+-------------+
|         161|       140167|
|         237|       140112|
|         132|       138304|
|         236|       133966|
|         162|       104336|
|         230|       102951|
|         186|       102122|
|         142|       101762|
|         138|        87662|
|         239|        86498|
+------------+-------------+
only showing top 10 rows


In [47]:
# 5. Ticket promedio por dia de la semana
# dayofweek: 1=Dom, 2=Lun, 3=Mar, 4=Mie, 5=Jue, 6=Vie, 7=Sab
(df_limpio
    .groupBy(date_format("tpep_pickup_datetime", "EEEE").alias("dia"))
    .agg(
        count("*").alias("viajes"),
        f_round(avg("total_amount"), 2).alias("ticket_promedio")
    )
    .orderBy(col("ticket_promedio").desc())
    .show())

+---------+------+---------------+
|      dia|viajes|ticket_promedio|
+---------+------+---------------+
|   Monday|393213|          28.82|
| Thursday|416280|          27.71|
|  Tuesday|449225|          27.61|
|Wednesday|480797|           27.5|
|   Sunday|326851|          27.21|
|   Friday|396388|          27.16|
| Saturday|407257|          25.18|
+---------+------+---------------+



In [48]:
informe_diario = (
    df_limpio
    .withColumn("dia", dayofmonth("tpep_pickup_datetime"))
    .groupBy("dia", "payment_type")
    .agg(
        count("*").alias("viajes"),
        f_round(f_sum_("total_amount"), 2).alias("ingreso"),
        f_round(avg("total_amount"), 2).alias("ticket_promedio")
    )
)

(informe_diario
    .write
    .mode("overwrite")
    .partitionBy("dia")
    .parquet("02-data/tmp/informe_diario"))

print("Informe guardado en 02-data/tmp/informe_diario")
!ls 02-data/tmp/informe_diario/ | head -5

Informe guardado en 02-data/tmp/informe_diario
dia=1
dia=10
dia=11
dia=12
dia=13


In [49]:
spark.stop()
print("SparkSession cerrada")

SparkSession cerrada
